# 🗜️ Media Lite — Smart Control Center

**واجهة واحدة بـ5 Tabs:** Smart Compressor · Drive → Telegram · Telegram → Drive · Internet Downloader · Gemini Ready.

1. شغّل **① افتح الواجهة** مرة بعد توصيل الـRuntime.
2. اختار التاب واضبط **Quick Settings**. افتح **Advanced** بس لو محتاج إعدادات إضافية.
3. شغّل **② نفّذ الأداة المختارة**. النتائج والـProgress هتظهر تحت خلية التنفيذ.
4. **الإيقاف:** استخدم زر ⏹ الأصلي في Colab أو Runtime → Interrupt execution. لإعادة الاختيار استخدم التاب بدون تشغيل خلية الواجهة تاني.

**تنبيه نقل Telegram → Drive:** خيار الحذف بعد التحقق مفعّل افتراضيًا بناءً على تفضيلك، لكن العملية الافتراضية **معاينة فقط** ولا تحذف. في التنفيذ، حذف الرسائل نهائي وبعد التحقق من رفعها على Drive.

**Undo:** النسخة القديمة محفوظة على [GitHub Backup](https://github.com/abdullahsamirashour/gpt/tree/backup/media-lite-pre-tabs-2026-11-20/telegram-smart-compressor). ممكن نرجعها لـ main لو طلبت.

> الواجهة لا تحمّل المحركات أو تتصل بـTelegram أو Drive بمجرد فتحها؛ المحرك المطلوب فقط يبدأ عند تشغيل الخلية ②.

In [ ]:
#@title 🎛️ ① افتح Media Lite Smart Tabs { display-mode: "form" }
import importlib.util
import json
import os
from pathlib import Path
import re
import subprocess
import sys
import urllib.request

_UI_REPO = "abdullahsamirashour/gpt"
_UI_REF = "main"  # use the same stable branch as the existing Colab link
_UI_FILE = "telegram-smart-compressor/smart_tabs_ui.py"
_UI_LOCAL = Path("/content/media_lite_smart_tabs_ui.py")
_UI_MARKER = Path("/content/media_lite_smart_tabs_ui.py.commit")

def _ui_ref_sha():
    proc = subprocess.run(
        ["git", "ls-remote", f"https://github.com/{_UI_REPO}.git",
         f"refs/heads/{_UI_REF}"],
        capture_output=True, text=True, timeout=20,
    )
    candidate = proc.stdout.strip().split()[0] if proc.returncode == 0 and proc.stdout.strip() else ""
    if re.fullmatch(r"[0-9a-f]{40}", candidate):
        return candidate
    req = urllib.request.Request(
        f"https://api.github.com/repos/{_UI_REPO}/git/ref/heads/{_UI_REF}",
        headers={"User-Agent": "Media-Lite-Tabs", "Cache-Control": "no-cache"},
    )
    with urllib.request.urlopen(req, timeout=30) as response:
        sha = json.load(response)["object"]["sha"]
    if not re.fullmatch(r"[0-9a-f]{40}", sha):
        raise RuntimeError("GitHub commit غير صالح.")
    return sha

_ui_sha = _ui_ref_sha()
if (not _UI_LOCAL.is_file() or not _UI_MARKER.is_file()
        or _UI_MARKER.read_text(encoding="ascii").strip() != _ui_sha):
    req = urllib.request.Request(
        f"https://raw.githubusercontent.com/{_UI_REPO}/{_ui_sha}/{_UI_FILE}",
        headers={"User-Agent": "Media-Lite-Tabs"},
    )
    with urllib.request.urlopen(req, timeout=45) as response:
        _ui_src = response.read().decode("utf-8")
    if len(_ui_src) < 10000:
        raise RuntimeError("Smart Tabs module لم يتحمّل كاملًا.")
    temp = _UI_LOCAL.with_suffix(".tmp")
    temp.write_text(_ui_src, encoding="utf-8")
    os.replace(temp, _UI_LOCAL)
    _UI_MARKER.write_text(_ui_sha, encoding="ascii")

_ui_spec = importlib.util.spec_from_file_location("media_lite_smart_tabs", _UI_LOCAL)
_ui_module = importlib.util.module_from_spec(_ui_spec)
sys.modules[_ui_spec.name] = _ui_module
_ui_spec.loader.exec_module(_ui_module)
MEDIA_LITE_UI = _ui_module.launch()
print("✅ Smart Tabs جاهزة. اختار التاب والإعدادات، ثم شغّل خلية ②.")


In [ ]:
#@title ▶ ② نفّذ الأداة المختارة (Stop من زر Colab الأصلي) { display-mode: "form" }
if "MEDIA_LITE_UI" not in globals():
    raise RuntimeError("شغّل خلية ① أولًا لفتح الـSmart Tabs.")
MEDIA_LITE_UI.execute_selected()


### 📖 ملاحظات سريعة

- **Compressor:** كل خيارات Advanced Audio وAdvanced Video محفوظة. قائمة الفيديو توضّح 720p/1080p وCRF وFPS الحقيقي لكل Preset؛ الحجم النهائي مش ثابت.
- **Drive ↔ Telegram:** الاستكمال والمعاينة والهاشتاج وإعادة رفع المحدد والفولدر الفرعي وQR Login كلها موجودة.
- **Internet Downloader:** نفس التحميل المتعدد والصيغ والوجهات. خيارات الـAudio/Video وTelegram تظهر حسب اختيارك لتقليل الزحمة.
- **Gemini Ready:** نفس مصادر الملفات والتقسيم وحد الحجم والمدة وAdvanced Audio والحذف الموثّق.
- **للطوارئ:** لو التابات ما ظهرتش أو واجهت مشكلة في Widgets، افتح [النسخة القديمة](https://colab.research.google.com/github/abdullahsamirashour/gpt/blob/backup/media-lite-pre-tabs-2026-11-20/telegram-smart-compressor/Smart_Compressor.ipynb) من غير أي تغيير في المحركات.